# **1. Engenharia de variáveis**

Este notebook transforma a tabela de partidas em uma linha de previsão por jogo em casa da Série A depois da pandemia. Cada variável usa apenas informação disponível antes do início do jogo, de modo que cada linha pode ser lida como "o que se sabia quando o próximo jogo em casa foi marcado". As variáveis são organizadas em seis grupos, depois retirados um de cada vez para medir quanto cada grupo contribui em cada clube.

| grupo | variáveis | pergunta |
|---|---|---|
| histórico | defasagens, médias móveis e nível da temporada do público do clube | quanto o público recente já informa? |
| calendário | dia da semana, horário, feriado, mês, rodada, dias de descanso | o momento do jogo no calendário importa? |
| clima | temperatura, chuva no horário e nas horas anteriores, umidade | calor e chuva afastam o torcedor? |
| momento | Elo, pontos e saldo nos últimos cinco jogos, posição e distâncias na tabela | a fase esportiva do clube atrai público? |
| adversário | Elo do adversário, poder de atrair público, clube grande, clássico | quem é o visitante? |
| estádio | estádio habitual, jogo fora da cidade, capacidade e público relativos do estádio | o clube trocou de estádio? |

In [1]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from pathlib import Path

import holidays
import numpy as np
import pandas as pd
from IPython.display import display

### **1.1. Configuração**

O rating Elo segue a atualização do World Football Elo, com vantagem de mando, multiplicador pela diferença de gols e retorno parcial à média entre temporadas. Ele começa em 2020, de modo que as duas temporadas da pandemia servem de aquecimento e os ratings já estão estáveis quando a série de público começa, em 2022.

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"

ELO_K = 20
ELO_HOME = 65
ELO_START = 1500
ELO_NEW_CLUB = 1450
ELO_SEASON_REVERSION = 0.2
FORM_WINDOW = 5
USUAL_VENUE_WINDOW = 19
BIG_CLUBS = {
    "Flamengo", "Corinthians", "Palmeiras", "São Paulo", "Vasco da Gama", "Fluminense",
    "Botafogo", "Grêmio", "Internacional", "Atlético-MG", "Cruzeiro", "Santos",
}

matches = pd.read_csv(PROCESSED / "matches.csv", parse_dates=["date", "kickoff_local"])
weather = pd.read_csv(INTERIM / "weather_hourly.csv.gz", parse_dates=["local_time"])
matches = matches.sort_values(["kickoff_local", "game_id"]).reset_index(drop=True)
played = matches["home_goals"].notna() & matches["away_goals"].notna()
print(f"partidas: {len(matches):,} | disputadas: {played.sum():,}")

partidas: 2,660 | disputadas: 2,640


## **2. Rating Elo**

Os ratings são atualizados partida a partida, em ordem de horário, sobre todas as partidas da Série A desde 2020. Um clube que entra na liga pela primeira vez começa abaixo da média, e todos os ratings são puxados 20% em direção a 1.500 no início de cada temporada. O rating guardado em cada linha é o anterior ao jogo.

In [3]:
def margin_multiplier(goal_difference: int) -> float:
    goal_difference = abs(goal_difference)
    if goal_difference <= 1:
        return 1.0
    if goal_difference == 2:
        return 1.5
    return (11 + goal_difference) / 8


ratings: dict[int, float] = {}
current_season = None
elo_home, elo_away = [], []
for row in matches.itertuples(index=False):
    if row.season != current_season:
        ratings = {club: (1 - ELO_SEASON_REVERSION) * value + ELO_SEASON_REVERSION * ELO_START for club, value in ratings.items()}
        current_season = row.season
    home_rating = ratings.setdefault(row.home_id, ELO_NEW_CLUB)
    away_rating = ratings.setdefault(row.away_id, ELO_NEW_CLUB)
    elo_home.append(home_rating)
    elo_away.append(away_rating)
    if pd.isna(row.home_goals) or pd.isna(row.away_goals):
        continue
    expected_home = 1 / (1 + 10 ** ((away_rating - home_rating - ELO_HOME) / 400))
    result_home = 1.0 if row.home_goals > row.away_goals else 0.5 if row.home_goals == row.away_goals else 0.0
    change = ELO_K * margin_multiplier(row.home_goals - row.away_goals) * (result_home - expected_home)
    ratings[row.home_id] = home_rating + change
    ratings[row.away_id] = away_rating - change

matches["elo_home"] = elo_home
matches["elo_away"] = elo_away
matches["elo_diff"] = matches["elo_home"] - matches["elo_away"]
display(pd.Series(ratings).rename(index=matches.drop_duplicates("home_id").set_index("home_id")["home"]).sort_values(ascending=False).head(10).round(0).to_frame("Elo atual"))

,Elo atual
Flamengo,1727.0
Palmeiras,1657.0
Cruzeiro,1618.0
Fluminense,1594.0
Atlético-MG,1527.0
Bahia,1521.0
Athletico-PR,1517.0
Santos,1516.0
Red Bull Bragantino,1507.0
Mirassol,1494.0


## **3. Momento e tabela**

Cada partida é desdobrada em uma linha por clube. Dentro da temporada, os pontos por jogo e o saldo de gols dos últimos cinco jogos medem o momento de curto prazo, e uma tabela acumulada fornece a posição, a distância para o líder e a distância para a zona de rebaixamento antes do jogo.

In [4]:
long = pd.concat([
    matches.loc[played, ["game_id", "season", "kickoff_local", "home_id", "home_goals", "away_goals"]].set_axis(["game_id", "season", "kickoff_local", "club_id", "goals_for", "goals_against"], axis=1),
    matches.loc[played, ["game_id", "season", "kickoff_local", "away_id", "away_goals", "home_goals"]].set_axis(["game_id", "season", "kickoff_local", "club_id", "goals_for", "goals_against"], axis=1),
]).sort_values(["kickoff_local", "game_id"])
long["points"] = np.select([long["goals_for"] > long["goals_against"], long["goals_for"] == long["goals_against"]], [3, 1], 0)
long["goal_difference"] = long["goals_for"] - long["goals_against"]
grouped = long.groupby(["club_id", "season"])
long["ppg_last5"] = grouped["points"].transform(lambda s: s.shift(1).rolling(FORM_WINDOW, min_periods=1).mean())
long["gd_last5"] = grouped["goal_difference"].transform(lambda s: s.shift(1).rolling(FORM_WINDOW, min_periods=1).sum())
long["ppg_season"] = grouped["points"].transform(lambda s: s.shift(1).expanding().mean())
long["won_last"] = grouped["points"].transform(lambda s: s.shift(1).eq(3).astype(float).where(s.shift(1).notna()))
form = long.set_index(["game_id", "club_id"])[["ppg_last5", "gd_last5", "ppg_season", "won_last"]]

table_rows = []
for season, frame in matches.groupby("season", sort=False):
    points, goal_difference, goals_for = defaultdict(int), defaultdict(int), defaultdict(int)
    clubs = set(frame["home_id"]) | set(frame["away_id"])
    for row in frame.itertuples(index=False):
        ranking = sorted(clubs, key=lambda c: (-points[c], -goal_difference[c], -goals_for[c]))
        position = {club: i + 1 for i, club in enumerate(ranking)}
        leader, relegation_line = points[ranking[0]], points[ranking[16]]
        for club in (row.home_id, row.away_id):
            table_rows.append({
                "game_id": row.game_id, "club_id": club, "position": position[club],
                "gap_to_leader": leader - points[club], "gap_to_relegation": points[club] - relegation_line,
            })
        if pd.isna(row.home_goals):
            continue
        h, a = int(row.home_goals), int(row.away_goals)
        points[row.home_id] += 3 if h > a else 1 if h == a else 0
        points[row.away_id] += 3 if a > h else 1 if h == a else 0
        goal_difference[row.home_id] += h - a
        goal_difference[row.away_id] += a - h
        goals_for[row.home_id] += h
        goals_for[row.away_id] += a
table = pd.DataFrame(table_rows).set_index(["game_id", "club_id"])

home_key = pd.MultiIndex.from_frame(matches[["game_id", "home_id"]])
for column in form.columns:
    matches[column] = form[column].reindex(home_key).to_numpy()
for column in table.columns:
    matches[column] = table[column].reindex(home_key).to_numpy()
matches.loc[matches["round"].eq(1), ["position", "gap_to_leader", "gap_to_relegation"]] = np.nan
display(matches.loc[matches["season"].eq(2025), ["round", "home", "elo_home", "ppg_last5", "position", "gap_to_leader", "gap_to_relegation"]].tail(5))

,round,home,elo_home,ppg_last5,position,gap_to_leader,gap_to_relegation
2275,38,Atlético-MG,1466.429378,0.4,14.0,34.0,3.0
2276,38,Internacional,1461.904146,1.0,18.0,38.0,-1.0
2277,38,Vitória,1466.790910,1.6,18.0,37.0,-1.0
2278,38,Ceará,1452.045455,0.8,17.0,36.0,0.0
2279,38,Sport,1305.798399,0.0,20.0,62.0,-26.0


## **4. Histórico de público**

O histórico de um clube é a sequência de seus públicos regulares em casa na Série A desde 2022, em ordem de horário; jogos restritos e sem público são ignorados, para que um jogo de portões fechados não derrube a média móvel. Para cada jogo em casa, as defasagens, a média móvel de cinco jogos, uma média com pesos exponenciais e o nível da temporada vêm apenas dos jogos anteriores. Um clube que volta à Série A usa suas últimas temporadas na Série A; um clube sem nenhum jogo na Série A desde 2022 começa sem histórico.

In [5]:
studied = matches["crowd_status"].ne("not_studied")
regular = matches["crowd_status"].eq("regular")
history_rows = []
for club, frame in matches.loc[studied].groupby("home_id"):
    previous = matches.loc[regular & matches["home_id"].eq(club)].sort_values("kickoff_local")
    times = previous["kickoff_local"].to_numpy()
    values = previous["attendance"].to_numpy(dtype=float)
    seasons = previous["season"].to_numpy()
    for row in frame.itertuples(index=False):
        n = int(np.searchsorted(times, np.datetime64(row.kickoff_local), side="left"))
        past = values[:n]
        last = past[-FORM_WINDOW:]
        same_season = past[seasons[:n] == row.season]
        previous_season = past[seasons[:n] == row.season - 1]
        weights = 0.7 ** np.arange(len(past))[::-1]
        history_rows.append({
            "game_id": row.game_id,
            "att_lag1": past[-1] if n >= 1 else np.nan,
            "att_lag2": past[-2] if n >= 2 else np.nan,
            "att_lag3": past[-3] if n >= 3 else np.nan,
            "att_ma5": last.mean() if n else np.nan,
            "att_ewm": (past * weights).sum() / weights.sum() if n else np.nan,
            "att_season_mean": same_season.mean() if len(same_season) else np.nan,
            "att_prev_season_mean": previous_season.mean() if len(previous_season) else np.nan,
            "att_history_n": n,
            "days_since_home": (row.kickoff_local - pd.Timestamp(times[n - 1])).days if n else np.nan,
        })
history = pd.DataFrame(history_rows)
matches = matches.merge(history, on="game_id", how="left")
display(matches.loc[studied, ["att_lag1", "att_ma5", "att_season_mean", "att_prev_season_mean"]].describe().round(0))

,att_lag1,att_ma5,att_season_mean,att_prev_season_mean
count,1850.0,1850.0,1780.0,1199.0
mean,25549.0,25510.0,24385.0,28220.0
std,15727.0,13728.0,13015.0,12806.0
min,1489.0,1577.0,1577.0,3765.0
25%,12798.0,15207.0,14680.0,20765.0
50%,23892.0,25557.0,24330.0,30132.0
75%,36700.0,34746.0,32442.0,34502.0
max,73244.0,70103.0,62342.0,62548.0


## **5. Calendário**

Dia da semana, horário e mês descrevem a posição do jogo no calendário. Feriados nacionais e estaduais vêm do pacote `holidays`, para o estado do estádio, junto com a véspera de feriado. Os dias de descanso e o número de jogos em casa nos 14 dias anteriores medem a sequência de jogos, que disputa o mesmo orçamento do torcedor.

In [6]:
years = sorted(matches["season"].unique().tolist() + [matches["season"].max() + 1])
holiday_sets = {state: holidays.country_holidays("BR", subdiv=state, years=years) for state in matches["state"].dropna().unique()}
national = holidays.country_holidays("BR", years=years)


def is_holiday(day, state) -> bool:
    return day in national or (isinstance(state, str) and day in holiday_sets.get(state, {}))


kickoff = matches["kickoff_local"]
matches["weekday"] = kickoff.dt.day_of_week
matches["weekend"] = matches["weekday"].ge(5).astype(int)
matches["kickoff_hour"] = kickoff.dt.hour + kickoff.dt.minute / 60
matches["month"] = kickoff.dt.month
matches["holiday"] = [int(is_holiday(d.date(), s)) if pd.notna(d) else np.nan for d, s in zip(kickoff, matches["state"])]
matches["holiday_eve"] = [int(is_holiday((d + pd.Timedelta(days=1)).date(), s)) if pd.notna(d) else np.nan for d, s in zip(kickoff, matches["state"])]
matches["final_rounds"] = matches["round"].ge(34).astype(int)

club_games = pd.concat([
    matches[["game_id", "kickoff_local", "home_id"]].set_axis(["game_id", "kickoff_local", "club_id"], axis=1).assign(at_home=1),
    matches[["game_id", "kickoff_local", "away_id"]].set_axis(["game_id", "kickoff_local", "club_id"], axis=1).assign(at_home=0),
]).dropna(subset=["kickoff_local"]).sort_values("kickoff_local")
club_games["rest_days"] = club_games.groupby("club_id")["kickoff_local"].diff().dt.total_seconds() / 86400
home_games = club_games[club_games["at_home"].eq(1)].set_index("kickoff_local")
home_games["home_games_14d"] = home_games.groupby("club_id")["at_home"].transform(lambda s: s.rolling("14D", closed="left").sum()).to_numpy()
matches = matches.merge(home_games.reset_index()[["game_id", "rest_days", "home_games_14d"]], on="game_id", how="left")
display(matches.loc[studied].groupby("weekday")["attendance"].median().rename(index=dict(enumerate("Mon Tue Wed Thu Fri Sat Sun".split()))).to_frame("público mediano"))

,público mediano
weekday,
Mon,16857.5
Tue,22411.0
Wed,18795.5
Thu,21400.5
Fri,34473.0
Sat,23861.0
Sun,26448.0


## **6. Clima**

A série horária da cidade-sede é unida no horário do jogo. A chuva é medida no horário do jogo e nas quatro horas anteriores, quando o torcedor decide se sai de casa. O arquivo histórico guarda o clima observado, que substitui a previsão do tempo que o torcedor teria visto no dia.

In [7]:
weather = weather.sort_values(["city", "state", "local_time"])
weather["rain_4h"] = weather.groupby(["city", "state"])["precipitation"].transform(lambda s: s.rolling(4, min_periods=1).sum())
matches["kickoff_hour_floor"] = matches["kickoff_local"].dt.floor("h")
matches = matches.merge(
    weather.rename(columns={"local_time": "kickoff_hour_floor", "precipitation": "rain_kickoff"})[["city", "state", "kickoff_hour_floor", "temperature", "rain_kickoff", "rain_4h", "humidity"]],
    on=["city", "state", "kickoff_hour_floor"], how="left",
).drop(columns="kickoff_hour_floor")
matches["rainy"] = (matches["rain_4h"] >= 1.0).astype(float).where(matches["rain_4h"].notna())
print(f"jogos estudados sem clima: {matches.loc[studied, 'temperature'].isna().sum()}")
display(matches.loc[studied].groupby("rainy")["attendance"].median().to_frame("público mediano"))

jogos estudados sem clima: 10


,público mediano
rainy,
0.0,23234.0
1.0,21253.0


## **7. Adversário**

O Elo do adversário mede sua qualidade, e os indicadores de clube grande e de clássico capturam a torcida visitante e a rivalidade. O poder de atrair público de um visitante é a média do logaritmo da razão entre o público que ele atraiu e a média móvel do mandante, apenas nas visitas anteriores.

In [8]:
home_state = matches.loc[studied].groupby("home_id")["state"].agg(lambda s: s.mode().iat[0])
home_city = matches.loc[studied].groupby("home_id")["city"].agg(lambda s: s.mode().iat[0])
matches["away_big_club"] = matches["away"].isin(BIG_CLUBS).astype(int)
matches["derby_state"] = (matches["home_id"].map(home_state) == matches["away_id"].map(home_state)).astype(int)
matches["derby_city"] = (matches["home_id"].map(home_city) == matches["away_id"].map(home_city)).astype(int)

matches["log_ratio"] = np.log(matches["attendance"] / matches["att_ma5"]).where(regular)
matches["away_pull"] = matches.groupby("away_id")["log_ratio"].transform(lambda s: s.shift(1).expanding().mean())
matches["away_pull_n"] = matches.groupby("away_id")["log_ratio"].transform(lambda s: s.shift(1).expanding().count())
display(matches.loc[matches["season"].eq(2026)].groupby("away")["away_pull"].last().sort_values(ascending=False).round(2).head(8).to_frame("poder de atrair público"))

,poder de atrair público
away,
Flamengo,0.37
Palmeiras,0.21
Corinthians,0.17
Santos,0.12
São Paulo,0.11
Vasco da Gama,0.05
Fluminense,0.00
Remo,-0.01


## **8. Estádio**

O estádio habitual de um clube é o mais frequente entre seus 19 jogos em casa anteriores, de modo que uma mudança temporária (enchente, reforma, troca de gramado ou show) fica marcada enquanto dura, e uma mudança permanente vira o novo estádio habitual depois de meia temporada. A cidade-sede habitual é definida da mesma forma.

Duas variáveis descrevem o estádio em relação ao clube e, por isso, não carregam informação sobre o tamanho do clube, que pertence ao grupo de histórico. A **razão de capacidade** compara o maior público já registrado no estádio com o maior público registrado no estádio habitual do clube (uma aproximação das capacidades). A **razão do estádio** compara o público médio do clube em seus jogos anteriores naquele estádio com sua média móvel: o Vasco leva no Maracanã muito mais que sua média em São Januário.

In [9]:
venue_rows = []
for club, frame in matches.loc[studied].groupby("home_id"):
    keys, cities = [], []
    for row in frame.sort_values("kickoff_local").itertuples(index=False):
        recent_keys, recent_cities = keys[-USUAL_VENUE_WINDOW:], cities[-USUAL_VENUE_WINDOW:]
        usual_key = Counter(recent_keys).most_common(1)[0][0] if recent_keys else row.venue_key
        usual_city = Counter(recent_cities).most_common(1)[0][0] if recent_cities else row.city
        venue_rows.append({
            "game_id": row.game_id,
            "usual_venue_key": usual_key,
            "venue_usual": int(row.venue_key == usual_key),
            "venue_out_of_city": int(row.city != usual_city),
            "venue_first_time": int(bool(keys) and row.venue_key not in keys),
        })
        keys.append(row.venue_key)
        cities.append(row.city)
matches = matches.merge(pd.DataFrame(venue_rows), on="game_id", how="left")

venue_max, venue_club_mean = [], []
best_at_venue, club_venue_values = {}, defaultdict(list)
for row in matches.itertuples(index=False):
    venue_max.append(best_at_venue.get(row.venue_key, np.nan))
    previous = club_venue_values[(row.home_id, row.venue_key)]
    venue_club_mean.append(np.mean(previous[-FORM_WINDOW:]) if previous else np.nan)
    if row.crowd_status == "regular":
        best_at_venue[row.venue_key] = max(best_at_venue.get(row.venue_key, 0), row.attendance)
        previous.append(row.attendance)
matches["venue_max_prior"] = venue_max
matches["venue_club_ma5"] = venue_club_mean
usual_max = []
best_at_venue = {}
for row in matches.itertuples(index=False):
    usual_max.append(best_at_venue.get(row.usual_venue_key, np.nan) if isinstance(row.usual_venue_key, str) else np.nan)
    if row.crowd_status == "regular":
        best_at_venue[row.venue_key] = max(best_at_venue.get(row.venue_key, 0), row.attendance)
matches["venue_capacity_ratio"] = (matches["venue_max_prior"] / pd.Series(usual_max, index=matches.index)).where(matches["venue_usual"].eq(0), 1.0)
matches["venue_ratio"] = (matches["venue_club_ma5"] / matches["att_ma5"]).where(matches["venue_usual"].eq(0), 1.0)
display(matches.loc[studied & matches["venue_usual"].eq(0)].groupby(["season", "home", "stadium"]).size().rename("jogos fora do estádio habitual").reset_index().sort_values(["season", "home"]).tail(20))

,season,home,stadium,jogos fora do estádio habitual
56,2025,Palmeiras,Arena Barueri,2
57,2025,Red Bull Bragantino,Cícero de Souza Marques,10
58,2025,Santos,Morumbi,2
59,2025,Santos,Nubank Parque,1
60,2025,Sport,Arena Pernambuco,1
61,2025,São Paulo,Vila Belmiro,4
62,2025,Vasco da Gama,Mané Garrincha,2
63,2025,Vasco da Gama,Maracanã,2
64,2026,Botafogo,Mané Garrincha,1
65,2026,Corinthians,Vila Belmiro,1


## **9. Exportação e auditoria**

| arquivo | conteúdo |
|---|---|
| `data/processed/features.csv` | uma linha por jogo em casa depois da pandemia, com alvo, situação do público e todas as variáveis |
| `data/processed/feature_groups.json` | as colunas de cada grupo de variáveis, lidas pelos notebooks de modelagem |

A auditoria verifica que nenhuma variável de um jogo regular usa o próprio público e que cada grupo existe na tabela.

In [10]:
FEATURE_GROUPS = {
    "history": ["att_lag1", "att_lag2", "att_lag3", "att_ma5", "att_ewm", "att_season_mean", "att_prev_season_mean", "att_history_n", "days_since_home"],
    "calendar": ["weekday", "weekend", "kickoff_hour", "month", "holiday", "holiday_eve", "round", "final_rounds", "rest_days", "home_games_14d"],
    "weather": ["temperature", "rain_kickoff", "rain_4h", "rainy", "humidity"],
    "form": ["elo_home", "elo_diff", "ppg_last5", "gd_last5", "ppg_season", "won_last", "position", "gap_to_leader", "gap_to_relegation"],
    "opponent": ["elo_away", "away_pull", "away_pull_n", "away_big_club", "derby_state", "derby_city"],
    "venue": ["venue_usual", "venue_out_of_city", "venue_first_time", "venue_capacity_ratio", "venue_ratio"],
}
id_columns = ["game_id", "season", "round", "date", "kickoff_local", "home_id", "home", "away_id", "away", "stadium", "city", "state", "attendance", "crowd_status"]
features = matches.loc[studied, id_columns + [c for group in FEATURE_GROUPS.values() for c in group if c not in id_columns]]
features.to_csv(PROCESSED / "features.csv", index=False)
(PROCESSED / "feature_groups.json").write_text(json.dumps(FEATURE_GROUPS, indent=2))

check = features[features["crowd_status"].eq("regular") & features["att_lag1"].notna()]
display(pd.Series({
    "linhas": len(features),
    "alvos_regulares": int(features["crowd_status"].eq("regular").sum()),
    "sem_publico": int(features["crowd_status"].eq("missing").sum()),
    "com_restricao": int(features["crowd_status"].eq("restricted").sum()),
    "linhas_sem_historico": int(features["att_history_n"].eq(0).sum()),
    "jogos_fora_do_estadio_habitual": int(features["venue_usual"].eq(0).sum()),
}, name="valor").to_frame())
assert not (check["att_lag1"] == check["attendance"]).all(), "A primeira defasagem repete o alvo"
assert features["game_id"].is_unique, "game_id duplicado"
assert set(FEATURE_GROUPS["history"]).issubset(features.columns), "Faltam colunas de histórico"
assert features[FEATURE_GROUPS["form"]].notna().any().all(), "Uma variável de momento está vazia"

,valor
linhas,1880
alvos_regulares,1675
sem_publico,203
com_restricao,2
linhas_sem_historico,30
jogos_fora_do_estadio_habitual,138
